## Imports & Connection

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'

In [2]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [3]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)
local_datasets = '/Users/rapido/local-datasets/customer/appography/appography_v2/bangalore/'
print(local_datasets)

data_dumper_path = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'

/Users/rapido/analytics/customer/Appography/Appography_v2
/Users/rapido/local-datasets/customer/appography/appography_v2/bangalore/


## Parameter & Datasets

In [4]:
start_date = '20240301' 
end_date = '20240331'
city = 'Bangalore'

### City Base Customer

- City-wise android fare estimate customers taked from the recent month to built the datasets.

- Criteria 

        - Andrioid Customer
        - Minimum 1 FE in recent month

city_fe_customer_query = f"""

        WITH fe_customer AS (

        SELECT
            user_id as user_id
        FROM
            hive.pricing.fare_estimates_enriched
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND platform = 'android'
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId,
            CASE WHEN age BETWEEN 15 AND 60 THEN age END age,
            gender
        FROM 
            datasets.customer_single_view
        )

        SELECT
            user_id,
            age,
            gender
        FROM 
            fe_customer
        LEFT JOIN 
            demographic_data
            on user_id = customerId        
"""

df_city_fe_customer = pd.read_sql(city_fe_customer_query, connection)
df_city_fe_customer.to_csv(local_datasets + '{}_city_fe_customer_{}_{}.csv'.format(city,start_date, end_date), index=False)

In [5]:
df_city_fe_customer = pd.read_csv(local_datasets + '{}_city_fe_customer_{}_{}.csv'.format(city,start_date, end_date))
print(df_city_fe_customer.head(3))
df_city_fe_customer.info()

                    user_id   age  gender
0  637f6762394b31a2642150d1   NaN  FEMALE
1  6605200258c88b36c39f5387   NaN    MALE
2  6466641674d89382040f2c87  26.0    MALE
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2285299 entries, 0 to 2285298
Data columns (total 3 columns):
 #   Column   Dtype  
---  ------   -----  
 0   user_id  object 
 1   age      float64
 2   gender   object 
dtypes: float64(1), object(2)
memory usage: 52.3+ MB


### Use-case

In [6]:
"""
df_customers_use_case = pd.read_csv('/Users/rapido/local-datasets/customer/appography/demographic_personas/processed/pan_customer_use_case_{}_{}.csv'\
                                .format(start_date, end_date))
df_customers_use_case = df_customers_use_case[['user_id', 
                                               'first_half_drop_primary','first_half_drop_secondary',
                                               'second_half_pickup_primary', 'second_half_pickup_secondary']]
df_customers_use_case.info()
"""

"\ndf_customers_use_case = pd.read_csv('/Users/rapido/local-datasets/customer/appography/demographic_personas/processed/pan_customer_use_case_{}_{}.csv'                                .format(start_date, end_date))\ndf_customers_use_case = df_customers_use_case[['user_id', \n                                               'first_half_drop_primary','first_half_drop_secondary',\n                                               'second_half_pickup_primary', 'second_half_pickup_secondary']]\ndf_customers_use_case.info()\n"

### Base Customers Recent Appograpgy Data

- Read the local appography data dumper and filter the data by selected city customers

In [7]:
start_time = time.time()

csv_files = [file for file in os.listdir(data_dumper_path) if file.endswith('.csv')]

dfs = []

# Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    file_path = os.path.join(data_dumper_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

# Concatenate all DataFrames in the list into a single DataFrame
df_combined_data_dumper = pd.concat(dfs, ignore_index=True)
end_time = time.time()
execution_time = ((end_time - start_time)/60.00)
print("Execution time : ", execution_time, " mins")
df_combined_data_dumper.head(5)

Execution time :  3.4824527700742087  mins


,last_sync_date,user_id,app_list_set
0,20240324,573f28f69b0ffc283676ff35,"['axis mobile', 'microsoft teams', 'whatsapp',..."
1,20240324,573f28f69b0ffc283676ff57,"['whatsapp', 'google photos', 'truecaller', 'a..."
2,20240324,573f28f79b0ffc2836770a38,"['onedrive', 'youtube', 'telegram', 'jio cinem..."
3,20240324,573f28f89b0ffc2836770daf,"['zomato', 'gmail', 'blusmart', 'google maps',..."
4,20240324,573f28f99b0ffc2836771403,"['cleartrip', 'google photos', 'linkedin', 'go..."


#### Optimization & work-around

In [8]:
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31723947 entries, 0 to 31723946
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int64 
 1   user_id         object
 2   app_list_set    object
dtypes: int64(1), object(2)
memory usage: 726.1+ MB


In [9]:
df_combined_data_dumper['last_sync_date'] = df_combined_data_dumper['last_sync_date'].astype('int32')
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31723947 entries, 0 to 31723946
Data columns (total 3 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   last_sync_date  int32 
 1   user_id         object
 2   app_list_set    object
dtypes: int32(1), object(2)
memory usage: 605.1+ MB


In [10]:
## Latest appography sync data
df_combined_data_dumper = df_combined_data_dumper.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])
df_combined_data_dumper = df_combined_data_dumper.drop_duplicates(subset='user_id', keep='first')
df_combined_data_dumper = df_combined_data_dumper[['user_id', 'app_list_set']]
df_combined_data_dumper.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17829492 entries, 25248151 to 18082310
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   user_id       object
 1   app_list_set  object
dtypes: object(2)
memory usage: 408.1+ MB


### Merge 

In [11]:
df_city_fe_customer.head(2)

,user_id,age,gender
0,637f6762394b31a2642150d1,NaN,FEMALE
1,6605200258c88b36c39f5387,NaN,MALE


In [12]:
# df_customers_use_case.head(2)

In [13]:
df_combined_data_dumper.rename(columns = {'user_id' : 'customer_id'}, inplace = True)
df_combined_data_dumper.head(2)

,customer_id,app_list_set
25248151,5737c6acddbec2203f73315e,"['flipkart', 'disney+ hotstar', 'messenger', '..."
23953873,5737c6adddbec2203f73316d,"['whatsapp', 'paytm money', 'samsung galaxy we..."


In [14]:
"""
df_city_fe_customer = pd.merge(df_city_fe_customer,
                               df_customers_use_case,
                               how='left',
                               on = 'user_id'
                              )
df_city_fe_customer.head(2)  
"""

"\ndf_city_fe_customer = pd.merge(df_city_fe_customer,\n                               df_customers_use_case,\n                               how='left',\n                               on = 'user_id'\n                              )\ndf_city_fe_customer.head(2)  \n"

In [15]:
## Merge City Base Customer & Latest appography sync data
df_base_customers_app_data = pd.merge(df_city_fe_customer,
                                      df_combined_data_dumper,
                                      how = 'inner',
                                      left_on = ['user_id'],
                                      right_on = ['customer_id']
                                     )
df_base_customers_app_data.head(1)

,user_id,age,gender,customer_id,app_list_set
0,637f6762394b31a2642150d1,NaN,FEMALE,637f6762394b31a2642150d1,"['google photos', 'cred', 'whatsapp', 'onedriv..."


In [16]:
df_base_customers_app_data[['user_id', 'age', 'gender', 'customer_id', 'app_list_set'
#                             ,'first_half_drop_primary', 'first_half_drop_secondary',
#                             'second_half_pickup_primary','second_half_pickup_secondary'
                           ]]\
.to_csv(local_datasets + '{}_city_fe_customer_with_appo_data.csv'.format(city), index=False)

In [17]:
df_base_customers_app_data = pd.read_csv(local_datasets + '{}_city_fe_customer_with_appo_data.csv'.format(city))

In [18]:
df_base_customers_app_data['city'] = city

In [19]:
print('Users without age values' ,(df_base_customers_app_data[df_base_customers_app_data['age'].isna()]['user_id'].nunique()*100.00/df_base_customers_app_data['user_id'].nunique()))
print('Users with age values' ,(df_base_customers_app_data[~df_base_customers_app_data['age'].isna()]['user_id'].nunique()*100.00/df_base_customers_app_data['user_id'].nunique()))

Users without age values 66.88718702109409
Users with age values 33.11281297890591


### Mobile app to category mapping

In [20]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['categories_l1'] = category_list['categories_l1'].str.lower()
category_list['categories_l2'] = category_list['categories_l2'].str.lower()
category_list = category_list[['app_name', 'categories_l1', 'categories_l2']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(399, 3)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app']
['educational' 'rest' 'office' 'ride haling' 'gaming' 'finance_investment'
 'driver_app']


,app_name,categories_l1,categories_l2
0,ignou e-content,educational,educational


In [21]:
core_education = ['aakash','chegg study','google classroom','microsoft math solver','simplilearn','moodle','vedantu','vidyakul','diksha','e-pg pathshala','mycbseguide','physics wallah']
core_office = ['slack','asana','confluence','jira','cisco','github','google analytics','miro','trello','zoho mail','zoho meeting']
core_social = ['hike','facebook','messenger','instagram','linkedin','moj','pinterest','skype','snapchat','twitter','whatsapp','tiktok','josh','sharechat']

category_list['categories_l3'] = np.where(category_list['app_name'].isin(core_education), 'core_education',  #category_list['app_name'],
                                 np.where(category_list['app_name'].isin(core_office), 'core_office', #category_list['app_name'],
                                 np.where(category_list['app_name'].isin(core_social), 'core_social',
                                 np.where(category_list['app_name'].isin(['intune  company portal']), 'intune  company portal',         
                                 category_list['categories_l1']))))

## Explode mobile app's and mapping categories 

In [22]:
df_customers_with_appography = df_base_customers_app_data[~df_base_customers_app_data['age'].isna()]
df_customers_with_appography.head(4)

,user_id,age,gender,customer_id,app_list_set,city
2,6466641674d89382040f2c87,26.0,MALE,6466641674d89382040f2c87,"['booking.com', 'google slides', 'cred', 'micr...",Bangalore
4,5c7d28ba875c98536260f476,28.0,FEMALE,5c7d28ba875c98536260f476,"['meesho', 'paytm money', 'namma yatri', 'uber...",Bangalore
6,5e675f15b9fe96fa9379cdac,26.0,FEMALE,5e675f15b9fe96fa9379cdac,"['webex', 'namma yatri', 'meesho', 'youtube', ...",Bangalore
8,6019087c73262d088dc9b5d0,27.0,FEMALE,6019087c73262d088dc9b5d0,"['netflix', 'gmail', 'namma yatri', 'voot', 'i...",Bangalore


In [23]:
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customers_with_appography['app_list_set'] = df_customers_with_appography['app_list_set'].apply(lambda x: extract_and_trim(x))
df_customers_with_appography = df_customers_with_appography[['city', 'user_id', 'age', 'gender', 
#                                                              'first_half_drop_primary', 'first_half_drop_secondary',
#                                                              'second_half_pickup_primary','second_half_pickup_secondary',
                                                             'app_list_set']]
df_customers_with_appography.head(4)

,city,user_id,age,gender,app_list_set
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,"[booking.com, google slides, cred, microsoft t..."
4,Bangalore,5c7d28ba875c98536260f476,28.0,FEMALE,"[meesho, paytm money, namma yatri, uber, nykaa..."
6,Bangalore,5e675f15b9fe96fa9379cdac,26.0,FEMALE,"[webex, namma yatri, meesho, youtube, whatsapp..."
8,Bangalore,6019087c73262d088dc9b5d0,27.0,FEMALE,"[netflix, gmail, namma yatri, voot, instagram,..."


In [24]:
## Explode list into individual rows

df_customers_with_appography_explode = df_customers_with_appography.explode('app_list_set')
df_customers_with_appography_explode['app_list_set'] = df_customers_with_appography_explode['app_list_set'].str.lower()

df_customers_with_appography_explode.head(4)

,city,user_id,age,gender,app_list_set
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,booking.com
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,google slides
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,cred
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,microsoft teams


### Merge

In [25]:
category_list.head(2)

,app_name,categories_l1,categories_l2,categories_l3
0,ignou e-content,educational,educational,educational
1,aha,ott,rest,ott


In [26]:
df_customers_app_category =  pd.merge(df_customers_with_appography_explode,
                                      category_list, 
                                      how='left', 
                                      left_on='app_list_set',  
                                      right_on='app_name'
                                     )
df_customers_app_category.head(4)

,city,user_id,age,gender,app_list_set,app_name,categories_l1,categories_l2,categories_l3
0,Bangalore,6466641674d89382040f2c87,26.0,MALE,booking.com,booking.com,travel_bookings,rest,travel_bookings
1,Bangalore,6466641674d89382040f2c87,26.0,MALE,google slides,google slides,tools,rest,tools
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,cred,cred,finance_transactions,rest,finance_transactions
3,Bangalore,6466641674d89382040f2c87,26.0,MALE,cred,cred,finance_transactions,rest,finance_transactions


#### Distribution check on categories

In [27]:
total_customers = df_customers_with_appography.user_id.nunique()
total_customers

647780

In [28]:
df_temp = df_customers_app_category\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('user_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,tools,"[picsart, google photos, notion, qr code reade...",647761,100.00
1,social,"[instagram, snapchat, telegram, flipboard, mes...",647012,99.88
2,ott,"[zee5, disney+ hotstar, sonyliv, amazon prime ...",552142,85.24
3,ecommerce,"[jiomart, flipkart, firstcry, amazon shopping,...",531312,82.02
4,finance_transactions,"[mobikwik, axis mobile, paytm money, freecharg...",512752,79.16
5,travel_ridehailing,"[ola, namma yatri, quick ride, blusmart, drive...",499397,77.09
6,delivery_food,"[uber eats, dunzo, swiggy, zomato]",454635,70.18
7,streaming_music,"[spotify, soundcloud, amazon music, gaana, wyn...",445790,68.82
8,office,"[outlook, zoho meeting, confluence, zoho mail,...",406087,62.69
9,finance_investment,"[kite by zerodha, angel broking, namma partner...",297696,45.96


In [29]:
df_temp = df_customers_app_category\
            .groupby(['categories_l2'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('user_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list,customers,%
0,rest,"[indigo, instagram, mi fit, qr code reader, fe...",647778,100.00
1,ride haling,"[ola, namma yatri, quick ride, blusmart, drive...",499397,77.09
2,office,"[outlook, zoho meeting, confluence, zoho mail,...",406087,62.69
3,finance_investment,"[kite by zerodha, angel broking, namma partner...",297696,45.96
4,educational,"[goodreads, diksha, coursera, vidyakul, mycbse...",131505,20.30
5,driver_app,"[zomato driver app, uber driver app, porter dr...",120653,18.63
6,gaming,[ludo king],93745,14.47


In [30]:
df_temp = df_customers_app_category[~df_customers_app_category['categories_l2'].isin(['rest'])]\
            .groupby(['categories_l2','app_list_set'])\
            .agg(customers = ('user_id','nunique'))\
            .sort_values(by=['categories_l2', 'customers'], ascending=[False, False])\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l2,app_list_set,customers,%
0,ride haling,ola,399708,61.70
1,ride haling,uber,354065,54.66
2,ride haling,namma yatri,143646,22.18
3,ride haling,quick ride,23449,3.62
4,ride haling,in drive,22155,3.42
5,ride haling,blusmart,20642,3.19
6,ride haling,driveu driver app,4163,0.64
7,ride haling,quickride,1694,0.26
8,ride haling,jugnoo,899,0.14
9,office,outlook,179867,27.77


## Transform Categories 

In [31]:
df_customers_app_category.head(4)

,city,user_id,age,gender,app_list_set,app_name,categories_l1,categories_l2,categories_l3
0,Bangalore,6466641674d89382040f2c87,26.0,MALE,booking.com,booking.com,travel_bookings,rest,travel_bookings
1,Bangalore,6466641674d89382040f2c87,26.0,MALE,google slides,google slides,tools,rest,tools
2,Bangalore,6466641674d89382040f2c87,26.0,MALE,cred,cred,finance_transactions,rest,finance_transactions
3,Bangalore,6466641674d89382040f2c87,26.0,MALE,cred,cred,finance_transactions,rest,finance_transactions


In [32]:
df_customer_mapped = df_customers_app_category\
                        .groupby(['city', 'user_id', 'age'
#                                   ,'first_half_drop_primary', 'first_half_drop_secondary',
#                                   'second_half_pickup_primary','second_half_pickup_secondary'
                                 ])\
                        .agg(
                             app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l3 = ('categories_l3', lambda x: list(set(x)))
                            )\
                        .reset_index()

df_customer_mapped.head(4)

,city,user_id,age,app_list,categories_l3
0,Bangalore,5737c6aeddbec2203f733176,34.0,"[ola, instagram, google photos, outlook, youtu...","[core_social, delivery_food, delivery_grocery,..."
1,Bangalore,5737c6b0ddbec2203f733182,28.0,"[ola, instagram, google photos, ola driver app...","[core_social, ecommerce, streaming_music, deli..."
2,Bangalore,5737c6b1ddbec2203f73318b,33.0,"[ola, instagram, google photos, grofers, youtu...","[core_social, ecommerce, travel_bookings, deli..."
3,Bangalore,5737c6c5ddbec2203f733239,28.0,"[instagram, google photos, youtube, hdfc mobil...","[core_social, ecommerce, streaming_music, fina..."


In [33]:
unique_categories = set(category for sublist in df_customer_mapped['categories_l3'] for category in sublist)
unique_categories_list = list(unique_categories)
print("Count of unique categories:", len(unique_categories))
sorted(unique_categories)

Count of unique categories: 27


['core_education',
 'core_office',
 'core_social',
 'dating',
 'delivery_food',
 'delivery_grocery',
 'devotional',
 'driver_app',
 'ecommerce',
 'educational',
 'entertainment',
 'fantasy_sports',
 'finance_investment',
 'finance_news',
 'finance_transactions',
 'gaming',
 'health',
 'intune  company portal',
 'news',
 'office',
 'ott',
 'social',
 'streaming_music',
 'tools',
 'travel_bookings',
 'travel_ridehailing',
 'wearable']

In [34]:
for category in unique_categories:
    df_customer_mapped[category] = df_customer_mapped['categories_l3'].apply(lambda x: 'yes' if category in x else 'no') 
    
df_customer_mapped.head(1)

,city,user_id,age,app_list,categories_l3,streaming_music,finance_transactions,ott,tools,social,educational,core_social,ecommerce,finance_news,finance_investment,devotional,dating,intune company portal,news,driver_app,travel_bookings,gaming,travel_ridehailing,health,core_education,delivery_food,core_office,wearable,entertainment,office,delivery_grocery,fantasy_sports
0,Bangalore,5737c6aeddbec2203f733176,34.0,"[ola, instagram, google photos, outlook, youtu...","[core_social, delivery_food, delivery_grocery,...",no,yes,yes,yes,no,no,yes,no,no,yes,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,yes,no


In [35]:
df_customer_mapped.columns

Index(['city', 'user_id', 'age', 'app_list', 'categories_l3',
       'streaming_music', 'finance_transactions', 'ott', 'tools', 'social',
       'educational', 'core_social', 'ecommerce', 'finance_news',
       'finance_investment', 'devotional', 'dating', 'intune  company portal',
       'news', 'driver_app', 'travel_bookings', 'gaming', 'travel_ridehailing',
       'health', 'core_education', 'delivery_food', 'core_office', 'wearable',
       'entertainment', 'office', 'delivery_grocery', 'fantasy_sports'],
      dtype='object')

In [36]:
bins = [15, 20, 25, 30, 35, 45, 60]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

df_customer_mapped['age_group'] = pd.cut(df_customer_mapped['age'], bins=bins, labels=labels, right=False)

In [37]:
main_list = ['city', 'user_id', 'age', 'age_group', 
#              'first_half_drop_primary','first_half_drop_secondary', 
#              'second_half_pickup_primary', 'second_half_pickup_secondary'
            ]
unique_clm_list = main_list + unique_categories_list

In [38]:
df_city_final_dataset = df_customer_mapped[unique_clm_list]

df_city_final_dataset.rename(columns = {'user_id' : 'customer_id'}, inplace=True)
df_city_final_dataset.head(5)

,city,customer_id,age,age_group,streaming_music,finance_transactions,ott,tools,social,educational,core_social,ecommerce,finance_news,finance_investment,devotional,dating,intune company portal,news,driver_app,travel_bookings,gaming,travel_ridehailing,health,core_education,delivery_food,core_office,wearable,entertainment,office,delivery_grocery,fantasy_sports
0,Bangalore,5737c6aeddbec2203f733176,34.0,31-35,no,yes,yes,yes,no,no,yes,no,no,yes,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,yes,no
1,Bangalore,5737c6b0ddbec2203f733182,28.0,26-30,yes,yes,yes,yes,yes,no,yes,yes,no,no,no,no,no,no,yes,no,no,yes,no,no,yes,no,no,yes,no,no,no
2,Bangalore,5737c6b1ddbec2203f73318b,33.0,31-35,no,yes,yes,yes,yes,no,yes,yes,no,no,no,no,no,yes,yes,yes,no,yes,no,no,yes,no,no,no,yes,yes,yes
3,Bangalore,5737c6c5ddbec2203f733239,28.0,26-30,yes,yes,yes,yes,yes,no,yes,yes,no,yes,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no
4,Bangalore,5737c6cfddbec2203f733299,42.0,36-45,no,yes,yes,yes,no,no,yes,yes,no,no,no,no,no,no,yes,no,yes,no,no,no,yes,no,no,no,no,no,no


#### Cross checking use-case with  intune company portal	& ms authenticator	

In [39]:
'''

df_city_final_dataset[(df_city_final_dataset['age_group'] == '15-20')
                      & 
                      ((df_city_final_dataset['intune  company portal'] == 'yes')
#                        |
#                        (df_city_final_dataset['ms authenticator'] == 'yes')
                      )]\
[['customer_id','age_group','first_half_drop_primary','first_half_drop_secondary',
  'second_half_pickup_primary','second_half_pickup_secondary',
  'intune  company portal']]\
.groupby(['first_half_drop_primary'])\
.agg(customers = ('customer_id', 'nunique'))


'''

"\n\ndf_city_final_dataset[(df_city_final_dataset['age_group'] == '15-20')\n                      & \n                      ((df_city_final_dataset['intune  company portal'] == 'yes')\n#                        |\n#                        (df_city_final_dataset['ms authenticator'] == 'yes')\n                      )][['customer_id','age_group','first_half_drop_primary','first_half_drop_secondary',\n  'second_half_pickup_primary','second_half_pickup_secondary',\n  'intune  company portal']].groupby(['first_half_drop_primary']).agg(customers = ('customer_id', 'nunique'))\n\n\n"

When customers under age_group '15-20' and installed 'intune  company portal' they've office as their use-case affinity

#### Agg

In [40]:
df_city_final_dataset.head(2)

,city,customer_id,age,age_group,streaming_music,finance_transactions,ott,tools,social,educational,core_social,ecommerce,finance_news,finance_investment,devotional,dating,intune company portal,news,driver_app,travel_bookings,gaming,travel_ridehailing,health,core_education,delivery_food,core_office,wearable,entertainment,office,delivery_grocery,fantasy_sports
0,Bangalore,5737c6aeddbec2203f733176,34.0,31-35,no,yes,yes,yes,no,no,yes,no,no,yes,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,yes,no
1,Bangalore,5737c6b0ddbec2203f733182,28.0,26-30,yes,yes,yes,yes,yes,no,yes,yes,no,no,no,no,no,no,yes,no,no,yes,no,no,yes,no,no,yes,no,no,no


In [41]:
df3_analysis = df_city_final_dataset\
                .groupby(['age_group'])\
                .agg(
                    dating = ('customer_id', lambda x: x[(df_city_final_dataset['dating'] == 'yes')].nunique()),
                    delivery_food = ('customer_id', lambda x: x[(df_city_final_dataset['delivery_food'] == 'yes')].nunique()),
                    delivery_grocery = ('customer_id', lambda x: x[(df_city_final_dataset['delivery_grocery'] == 'yes')].nunique()),
                    devotional = ('customer_id', lambda x: x[(df_city_final_dataset['devotional'] == 'yes')].nunique()),
                    driver_app = ('customer_id', lambda x: x[(df_city_final_dataset['driver_app'] == 'yes')].nunique()),
                    ecommerce = ('customer_id', lambda x: x[(df_city_final_dataset['ecommerce'] == 'yes')].nunique()),
                    core_education = ('customer_id', lambda x: x[(df_city_final_dataset['core_education'] == 'yes')].nunique()),                
                    educational = ('customer_id', lambda x: x[(df_city_final_dataset['educational'] == 'yes') & (df_city_final_dataset['intune  company portal'] == 'yes')].nunique()),
                    entertainment = ('customer_id', lambda x: x[(df_city_final_dataset['entertainment'] == 'yes')].nunique()),
                    fantasy_sports = ('customer_id', lambda x: x[(df_city_final_dataset['fantasy_sports'] == 'yes')].nunique()),
                    finance_investment = ('customer_id', lambda x: x[(df_city_final_dataset['finance_investment'] == 'yes')].nunique()),
                    finance_news = ('customer_id', lambda x: x[(df_city_final_dataset['finance_news'] == 'yes')].nunique()),
                    finance_transactions = ('customer_id', lambda x: x[(df_city_final_dataset['finance_transactions'] == 'yes')].nunique()),
                    gaming = ('customer_id', lambda x: x[(df_city_final_dataset['gaming'] == 'yes')].nunique()),
                    health = ('customer_id', lambda x: x[(df_city_final_dataset['health'] == 'yes')].nunique()),
                    news = ('customer_id', lambda x: x[(df_city_final_dataset['news'] == 'yes')].nunique()),
                    core_office = ('customer_id', lambda x: x[(df_city_final_dataset['core_office'] == 'yes') | (df_city_final_dataset['intune  company portal'] == 'yes')].nunique()),                
                    intune_company_portal = ('customer_id', lambda x: x[(df_city_final_dataset['intune  company portal'] == 'yes')].nunique()),                
                    office = ('customer_id', lambda x: x[(df_city_final_dataset['office'] == 'yes')].nunique()),
                    ott = ('customer_id', lambda x: x[(df_city_final_dataset['ott'] == 'yes')].nunique()),
                    core_social = ('customer_id', lambda x: x[(df_city_final_dataset['core_social'] == 'yes')].nunique()),
                    social = ('customer_id', lambda x: x[(df_city_final_dataset['social'] == 'yes')].nunique()),
                    streaming_music = ('customer_id', lambda x: x[(df_city_final_dataset['streaming_music'] == 'yes')].nunique()),
                    tools = ('customer_id', lambda x: x[(df_city_final_dataset['tools'] == 'yes')].nunique()),
                    travel_bookings = ('customer_id', lambda x: x[(df_city_final_dataset['travel_bookings'] == 'yes')].nunique()),
                    travel_ridehailing = ('customer_id', lambda x: x[(df_city_final_dataset['travel_ridehailing'] == 'yes')].nunique()),
                    wearable = ('customer_id', lambda x: x[(df_city_final_dataset['wearable'] == 'yes')].nunique())
                    )\
                .reset_index()

df3_analysis

,age_group,dating,delivery_food,delivery_grocery,devotional,driver_app,ecommerce,core_education,educational,entertainment,fantasy_sports,finance_investment,finance_news,finance_transactions,gaming,health,news,core_office,intune_company_portal,office,ott,core_social,social,streaming_music,tools,travel_bookings,travel_ridehailing,wearable
0,15-20,1097,16895,8587,0,3930,22230,3936,344,9006,1229,7622,358,17063,4571,1252,8788,2109,1249,13634,22484,28561,17464,20396,28621,6026,20068,2867
1,21-25,11946,106600,55497,3,28544,127555,19079,5488,60887,6297,64552,2856,121316,26406,8214,50793,28012,17118,88985,129564,152657,112536,112297,152838,48333,111327,16203
2,26-30,18795,158224,82800,3,37293,179988,7436,10687,90256,9028,109095,4688,173898,34521,14528,79201,54554,37243,135050,186748,213606,146870,149640,213925,83411,164933,30790
3,31-35,7692,95032,50599,4,26466,109387,3189,6226,54828,5953,66157,3072,107348,17707,9129,50604,31749,22949,84029,114440,131879,81389,88971,132118,52846,105516,20482
4,36-45,3723,63165,33282,3,20360,75057,3526,3510,32895,3719,41911,1979,75294,9017,5847,34249,18062,13750,60235,79619,95623,52187,60681,95832,33810,77276,14534
5,46-60,555,14468,7846,2,4019,16837,1146,428,6045,556,8226,492,17539,1505,1314,7279,2568,1770,15358,18965,23946,11129,13601,24002,7566,19904,3666


In [42]:
df3_analysis.to_clipboard(index=False)

In [43]:
local_datasets

'/Users/rapido/local-datasets/customer/appography/appography_v2/bangalore/'

In [44]:
df_city_final_dataset.to_csv(local_datasets + '{}_age_bins_refined_data.csv'.format(city), index=False)

## Analysis
### Age bucket

In [45]:
df_data = pd.read_csv(local_datasets + '{}_age_bins_refined_data.csv'.format(city))
df_data.head(5)

,city,customer_id,age,age_group,streaming_music,finance_transactions,ott,tools,social,educational,core_social,ecommerce,finance_news,finance_investment,devotional,dating,intune company portal,news,driver_app,travel_bookings,gaming,travel_ridehailing,health,core_education,delivery_food,core_office,wearable,entertainment,office,delivery_grocery,fantasy_sports
0,Bangalore,5737c6aeddbec2203f733176,34.0,31-35,no,yes,yes,yes,no,no,yes,no,no,yes,no,no,no,no,no,no,no,yes,no,no,yes,no,no,no,yes,yes,no
1,Bangalore,5737c6b0ddbec2203f733182,28.0,26-30,yes,yes,yes,yes,yes,no,yes,yes,no,no,no,no,no,no,yes,no,no,yes,no,no,yes,no,no,yes,no,no,no
2,Bangalore,5737c6b1ddbec2203f73318b,33.0,31-35,no,yes,yes,yes,yes,no,yes,yes,no,no,no,no,no,yes,yes,yes,no,yes,no,no,yes,no,no,no,yes,yes,yes
3,Bangalore,5737c6c5ddbec2203f733239,28.0,26-30,yes,yes,yes,yes,yes,no,yes,yes,no,yes,no,no,no,no,yes,no,no,no,no,no,no,no,no,no,no,no,no
4,Bangalore,5737c6cfddbec2203f733299,42.0,36-45,no,yes,yes,yes,no,no,yes,yes,no,no,no,no,no,no,yes,no,yes,no,no,no,yes,no,no,no,no,no,no


In [46]:
df1_analysis = df_data\
                .groupby(['age_group'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()
df1_analysis['% city threshold'] =  (df1_analysis['customers']*100.00/df1_analysis.customers.sum()).round(2)
df1_analysis

,age_group,customers,% city threshold
0,15-20,28623,4.42
1,21-25,152840,23.61
2,26-30,213932,33.05
3,31-35,132120,20.41
4,36-45,95836,14.80
5,46-60,24004,3.71


In [47]:
df2_analysis = df_data\
                .groupby(['age_group'])\
                .agg(
                    dating = ('customer_id', lambda x: x[(df_data['dating'] == 'yes')].nunique()),
                    delivery_food = ('customer_id', lambda x: x[(df_data['delivery_food'] == 'yes')].nunique()),
                    delivery_grocery = ('customer_id', lambda x: x[(df_data['delivery_grocery'] == 'yes')].nunique()),
                    devotional = ('customer_id', lambda x: x[(df_data['devotional'] == 'yes')].nunique()),
                    driver_app = ('customer_id', lambda x: x[(df_data['driver_app'] == 'yes')].nunique()),
                    ecommerce = ('customer_id', lambda x: x[(df_data['ecommerce'] == 'yes')].nunique()),
                    core_education = ('customer_id', lambda x: x[(df_data['core_education'] == 'yes')].nunique()),                
                    educational = ('customer_id', lambda x: x[(df_data['educational'] == 'yes')].nunique()),
                    entertainment = ('customer_id', lambda x: x[(df_data['entertainment'] == 'yes')].nunique()),
                    fantasy_sports = ('customer_id', lambda x: x[(df_data['fantasy_sports'] == 'yes')].nunique()),
                    finance_investment = ('customer_id', lambda x: x[(df_data['finance_investment'] == 'yes')].nunique()),
                    finance_news = ('customer_id', lambda x: x[(df_data['finance_news'] == 'yes')].nunique()),
                    finance_transactions = ('customer_id', lambda x: x[(df_data['finance_transactions'] == 'yes')].nunique()),
                    gaming = ('customer_id', lambda x: x[(df_data['gaming'] == 'yes')].nunique()),
                    health = ('customer_id', lambda x: x[(df_data['health'] == 'yes')].nunique()),
                    news = ('customer_id', lambda x: x[(df_data['news'] == 'yes')].nunique()),
                    core_office = ('customer_id', lambda x: x[(df_data['core_office'] == 'yes')].nunique()),                
                    office = ('customer_id', lambda x: x[(df_data['office'] == 'yes')].nunique()),
                    ott = ('customer_id', lambda x: x[(df_data['ott'] == 'yes')].nunique()),
                    core_social = ('customer_id', lambda x: x[(df_data['core_social'] == 'yes')].nunique()),
                    social = ('customer_id', lambda x: x[(df_data['social'] == 'yes')].nunique()),
                    streaming_music = ('customer_id', lambda x: x[(df_data['streaming_music'] == 'yes')].nunique()),
                    tools = ('customer_id', lambda x: x[(df_data['tools'] == 'yes')].nunique()),
                    travel_bookings = ('customer_id', lambda x: x[(df_data['travel_bookings'] == 'yes')].nunique()),
                    travel_ridehailing = ('customer_id', lambda x: x[(df_data['travel_ridehailing'] == 'yes')].nunique()),
                    wearable = ('customer_id', lambda x: x[(df_data['wearable'] == 'yes')].nunique())
                    )\
                .reset_index()

df2_analysis

,age_group,dating,delivery_food,delivery_grocery,devotional,driver_app,ecommerce,core_education,educational,entertainment,fantasy_sports,finance_investment,finance_news,finance_transactions,gaming,health,news,core_office,office,ott,core_social,social,streaming_music,tools,travel_bookings,travel_ridehailing,wearable
0,15-20,1097,16895,8587,0,3930,22230,3936,4960,9006,1229,7622,358,17063,4571,1252,8788,957,13634,22484,28561,17464,20396,28621,6026,20068,2867
1,21-25,11946,106600,55497,3,28544,127555,19079,30728,60887,6297,64552,2856,121316,26406,8214,50793,12605,88985,129564,152657,112536,112297,152838,48333,111327,16203
2,26-30,18795,158224,82800,3,37293,179988,7436,38177,90256,9028,109095,4688,173898,34521,14528,79201,20794,135050,186748,213606,146870,149640,213925,83411,164933,30790
3,31-35,7692,95032,50599,4,26466,109387,3189,20706,54828,5953,66157,3072,107348,17707,9129,50604,10982,84029,114440,131879,81389,88971,132118,52846,105516,20482
4,36-45,3723,63165,33282,3,20360,75057,3526,12941,32895,3719,41911,1979,75294,9017,5847,34249,5478,60235,79619,95623,52187,60681,95832,33810,77276,14534
5,46-60,555,14468,7846,2,4019,16837,1146,2554,6045,556,8226,492,17539,1505,1314,7279,918,15358,18965,23946,11129,13601,24002,7566,19904,3666


In [48]:
# df1_analysis.to_clipboard(index=False)

In [49]:
# df2_analysis.to_clipboard(index=False)

### App level vectorizer

dating<br>
delivery_food<br>
driver_app<br>
core_education<br>
entertainment<br>
finance_investment<br>
gaming<br>
core_office<br>
office<br>
social<br>
streaming_music<br>
travel_bookings<br>
travel_ridehailing<br>